# TerraAlert - Landslide Susceptibility Model Training

## Dataset
**NASA Global Landslide Catalog (GLC)**
- Download CSV từ: https://data.nasa.gov/Earth-Sciences/Global-Landslide-Catalog-Export/fnnf-nbjd
- Nếu link trên không hoạt động, search Google: `NASA Global Landslide Catalog download CSV`
- Upload CSV vào Kaggle Files (tab Data bên phải → Upload)

## Output
- `lsm_xgboost_model.pkl` - Model chính
- `lsm_random_forest_model.pkl` - Model backup  
- `label_encoders.pkl` - Encoders cho categorical features
- `model_metadata.json` - Metadata + metrics

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score, accuracy_score
from sklearn.inspection import permutation_importance

import xgboost as xgb
import joblib
import json
from datetime import datetime
import os

print('Libraries loaded!')

## Step 1: Load Dataset

### Cách lấy dữ liệu:

1. **Vào Kaggle:** https://www.kaggle.com/datasets?search=global+landslide
2. **Search:** `global landslide catalog` hoặc `landslide prediction`
3. **Chọn dataset** có nhiều downloads nhất (thường có >10K records)
4. **Add to Notebook:** Click vào dataset → "New Notebook" hoặc "Add to Notebook"

### Nếu không tìm thấy trên Kaggle:
1. Google: `NASA Global Landslide Catalog CSV download`
2. Download file CSV
3. Upload lên Kaggle: Tab **Files** → **Upload** → chọn file CSV

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

# ------------------------------------
# CÁCH 1: Upload CSV từ script create_dataset.py
# ------------------------------------
# 1. Chạy script local: python create_dataset.py --output landslide_dataset.csv
# 2. Upload landslide_dataset.csv lên Kaggle Files (tab Files → Upload)
# 3. Đổi CSV_PATH bên dưới thành path đúng

# ------------------------------------
# CÁCH 2: Add dataset từ Kaggle
# ------------------------------------
# 1. Search trên Kaggle: "global landslide" hoặc "Vietnam disaster"
# 2. Click dataset → Add to Notebook
# 3. Path sẽ là: /kaggle/input/ten-dataset/file.csv

# ------------------------------------
# CÁCH 3: Demo mode (synthetic data)
# ------------------------------------
# Nếu chưa có file, notebook sẽ chạy demo

import glob

# Tìm file CSV trong Kaggle
csv_candidates = [
    '/kaggle/input/landslide-dataset/landslide_dataset.csv',
    '/kaggle/input/landslide-dataset/*.csv',
    '/kaggle/working/landslide_dataset.csv',
    'landslide_dataset.csv',
]

raw_df = None
for pattern in csv_candidates:
    matches = glob.glob(pattern)
    if matches:
        csv_path = matches[0]
        raw_df = pd.read_csv(csv_path)
        print(f'Loaded: {csv_path}')
        print(f'Shape: {raw_df.shape}')
        print(f'Columns: {raw_df.columns.tolist()}')
        break

if raw_df is None:
    print('No CSV file found. Running in DEMO mode with synthetic data.')
    print('')
    print('To use real data:')
    print('  Option A: Run create_dataset.py locally, upload CSV to Kaggle Files')
    print('  Option B: Add dataset from Kaggle Search (global landslide)')
    print('  Option C: Download from HDX: https://data.humdata.org/ (search Vietnam disaster)')

In [ ]:
# ============================================================
# XỬ LÝ DATA
# ============================================================

if raw_df is not None:
    # Dataset từ create_dataset.py đã có sẵn format
    # Columns: latitude, longitude, landslide, elevation, slope, aspect,
    #          annual_precipitation, max_daily_rainfall, rainfall_intensity,
    #          ndvi, distance_to_fault
    
    df = raw_df.copy()
    
    # Đảm bảo có các cột cần thiết
    required_cols = ['latitude', 'longitude', 'landslide', 'elevation', 'slope', 'ndvi']
    missing = [c for c in required_cols if c not in df.columns]
    
    if missing:
        print(f'Missing columns: {missing}')
        print(f'Available: {df.columns.tolist()}')
    else:
        print(f'Dataset loaded successfully!')
        print(f'Shape: {df.shape}')
        print(f'Landslide distribution:\n{df["landslide"].value_counts()}')
        
        # Fill missing values
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
        
        # Thêm categorical features nếu chưa có
        if 'soil_type' not in df.columns:
            df['soil_type'] = np.random.choice(['clay', 'sand', 'silt', 'loam', 'rock'], len(df))
        if 'lithology' not in df.columns:
            df['lithology'] = np.random.choice(['sedimentary', 'igneous', 'metamorphic', 'alluvial'], len(df))
        if 'land_use' not in df.columns:
            df['land_use'] = np.random.choice(['forest', 'agriculture', 'urban', 'barren', 'grassland'], len(df))
        
else:
    # ========================================================
    # DEMO MODE: Synthetic data
    # ========================================================
    print('Running DEMO mode with synthetic data')
    
    np.random.seed(42)
    n = 5000
    
    df = pd.DataFrame({
        'latitude': np.random.uniform(10.0, 23.0, n),
        'longitude': np.random.uniform(103.0, 110.0, n),
        'elevation': np.abs(np.random.normal(400, 250, n)),
        'slope': np.abs(np.random.exponential(15, n)),
        'aspect': np.random.uniform(0, 360, n),
        'annual_precipitation': np.abs(np.random.normal(1800, 600, n)),
        'max_daily_rainfall': np.abs(np.random.normal(80, 30, n)),
        'rainfall_intensity': np.abs(np.random.normal(25, 10, n)),
        'ndvi': np.random.uniform(0.1, 0.7, n),
        'distance_to_fault': np.abs(np.random.exponential(5, n)),
        'soil_type': np.random.choice(['clay', 'sand', 'silt', 'loam', 'rock'], n),
        'lithology': np.random.choice(['sedimentary', 'igneous', 'metamorphic', 'alluvial'], n),
        'land_use': np.random.choice(['forest', 'agriculture', 'urban', 'barren', 'grassland'], n),
    })
    
    # Tạo label dựa trên risk factors
    risk = (
        (df['slope'] / 50) * 0.25 +
        (df['annual_precipitation'] / 3000) * 0.20 +
        (df['max_daily_rainfall'] / 150) * 0.15 +
        (1 / (df['distance_to_fault'] + 1)) * 0.20 +
        (1 - df['ndvi']) * 0.20
    )
    df['landslide'] = (risk + np.random.normal(0, 0.08, n) > 0.45).astype(int)

print(f'\nDataset: {df.shape}')
print(f'Landslide distribution:\n{df["landslide"].value_counts()}')
df.head()

In [ ]:
# Feature Engineering
le_soil = LabelEncoder()
le_litho = LabelEncoder()
le_landuse = LabelEncoder()

df['soil_encoded'] = le_soil.fit_transform(df['soil_type'])
df['lithology_encoded'] = le_litho.fit_transform(df['lithology'])
df['landuse_encoded'] = le_landuse.fit_transform(df['land_use'])

df['slope_rainfall'] = df['slope'] * df['annual_precipitation'] / 1000
df['elev_slope_ratio'] = df['elevation'] / (df['slope'] + 1)
df['terrain_rugged'] = df['elevation'] * df['slope'] / 100
df['rain_elev_idx'] = df['annual_precipitation'] * df['elevation'] / 10000
df['fault_river'] = 1/(df['distance_to_fault']+1) + 1/(df['distance_to_river']+1)

features = [
    'elevation', 'slope', 'aspect', 'curvature',
    'annual_precipitation', 'max_daily_rainfall', 'rainfall_intensity',
    'distance_to_fault', 'soil_encoded', 'lithology_encoded',
    'landuse_encoded', 'ndvi',
    'distance_to_river', 'drainage_density',
    'distance_to_road', 'population_density',
    'slope_rainfall', 'elev_slope_ratio', 'terrain_rugged', 'rain_elev_idx', 'fault_river'
]

X = df[features].fillna(0)
y = df['landslide']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train models
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)
rf_cv = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
print(f'RF CV ROC-AUC: {rf_cv.mean():.4f}')

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
    eval_metric='auc', use_label_encoder=False
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_cv = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
print(f'XGB CV ROC-AUC: {xgb_cv.mean():.4f}')

# Compare
print(f'\nTest ROC-AUC: RF={roc_auc_score(y_test, rf_proba):.4f}, XGB={roc_auc_score(y_test, xgb_proba):.4f}')

In [ ]:
# Export
os.makedirs('output', exist_ok=True)

metadata = {
    'model_name': 'TerraAlert_LSM',
    'version': '1.0.0',
    'trained_at': datetime.now().isoformat(),
    'features': features,
    'metrics': {
        'xgb_roc_auc': float(roc_auc_score(y_test, xgb_proba)),
        'xgb_f1': float(f1_score(y_test, xgb_pred)),
        'rf_roc_auc': float(roc_auc_score(y_test, rf_proba)),
    },
    'encoders': {
        'soil_type': le_soil.classes_.tolist(),
        'lithology': le_litho.classes_.tolist(),
        'land_use': le_landuse.classes_.tolist(),
    }
}

joblib.dump(xgb_model, 'output/lsm_xgboost_model.pkl')
joblib.dump(rf, 'output/lsm_random_forest_model.pkl')
joblib.dump({'soil_type': le_soil, 'lithology': le_litho, 'land_use': le_landuse}, 'output/label_encoders.pkl')
json.dump(metadata, open('output/model_metadata.json', 'w'), indent=2)

print('Saved to output/')
for f in os.listdir('output'):
    print(f'  {f}: {os.path.getsize(f"output/{f}")/1024:.0f} KB')

print(f'\nDONE! Download files from output/ and copy to backend/app/slow_lane/ml/models/')